In [1]:
import json
from pathlib import Path
from typing import Any

import pandas as pd
import wandb
import weight_formats.quantisation as Q

In [ ]:
DIR_PATH = Path("out/2025-10-01T17-03-17")
RUN_NAMES = [
    "avid-sun-950",
    "crisp-galaxy-954",
    "giddy-durian-952",
    "glowing-plasma-953",
    "silvery-pyramid-951",
    "valiant-dream-949",
]
WANDB_RUNS = wandb.Api().runs("graphcore/llama-mobile", filters={"config.name": "sweep-v0"})

In [3]:
def get_fmt(run_name: str) -> str | Any:
    for run in WANDB_RUNS:
        if run.name == run_name:
            fmt = run.config["quantisation"]["fmt"]
            return str(Q.TensorFormat.load(fmt))

def load_results(run_name: str, qat: bool) -> dict[str, Any]:
    if qat:
        path = DIR_PATH / f"{run_name}.json"
    else:
        path = DIR_PATH / f"{run_name}_dc.json"

    with open(path) as f:
        d = json.loads(f.read())
        d["qat"] = qat
        d["fmt"] = get_fmt(run_name)
        return d

In [4]:
baseline = load_results("baseline", True)
baseline["qat"] = False

In [5]:
df_base = pd.DataFrame.from_records([baseline])
display(df_base)

,accuracy,accuracy_easy,results,qat,fmt
0,0.754557,0.779297,"[{'id': 258996007, 'output': '5<|eot_id|><|eot...",False,None


In [6]:
df = pd.DataFrame.from_records(
    load_results(name, qat) for name in RUN_NAMES for qat in [False, True]
)
display(df)

,validation_loss,accuracy,accuracy_easy,results,qat,fmt
0,0.095557,0.727539,0.757161,"[{'id': 258996007, 'output': '5<|eot_id|><|eot...",False,"E0M3{1,128:BFLOAT16:absmax}"
1,0.027461,0.309570,0.723307,"[{'id': 258996007, 'output': 'There are five c...",True,"E0M3{1,128:BFLOAT16:absmax}"
2,0.550526,0.412435,0.681315,"[{'id': 258996007, 'output': 'There are three ...",False,"E0M2{1,64:BFLOAT16:absmax}"
3,0.057193,0.008464,0.392904,"[{'id': 258996007, 'output': 'The image shows ...",True,"E0M2{1,64:BFLOAT16:absmax}"
4,10.451298,0.000326,0.000977,"[{'id': 258996007, 'output': ' Minute.nettyUNU...",False,"E0M2{1,*:BFLOAT16:absmax}"
5,0.193887,0.001302,0.393880,"[{'id': 258996007, 'output': 'Answer: 3.<|eot_...",True,"E0M2{1,*:BFLOAT16:absmax}"
6,0.845449,0.282227,0.481445,"[{'id': 258996007, 'output': 'The image shows ...",False,"E0M2{1,128:BFLOAT16:absmax}"
7,0.068128,0.005534,0.380859,"[{'id': 258996007, 'output': 'The image shows ...",True,"E0M2{1,128:BFLOAT16:absmax}"
8,0.067720,0.749023,0.767578,"[{'id': 258996007, 'output': '5<|eot_id|><|eot...",False,"E0M3{1,64:BFLOAT16:absmax}"
9,0.024543,0.390951,0.763346,"[{'id': 258996007, 'output': 'There are five c...",True,"E0M3{1,64:BFLOAT16:absmax}"
